In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.0 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Ferrite and austenite on BEER

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Calibrate the BEER instrument at ESS with two McStas-simulated banks of a ferrite and austenite
duplex steel, fitted together: each bank has its own zero and resolution.

The project is the catalogue case `pd-neut-tof_ferrite-austenite-beer`, the one the command line
and the app's Examples open. Austenite's displacement follows ferrite's, and each phase's scale
in bank n2 follows bank s2, from the start.

In [2]:
import easydiffraction as edi

## Load the project

In [3]:
project_path = edi.download_data('pd-neut-tof_ferrite-austenite-beer', destination='data')
project = edi.Project.load(project_path)
ferrite = project.structures['ferrite']
austenite = project.structures['austenite']
bank_s2 = project.experiments['expt_s2']
bank_n2 = project.experiments['expt_n2']
project.display.constraints()

Data 'pd-neut-tof_ferrite-austenite-beer' is at data/pd-neut-tof_ferrite-austenite-beer-eccc89de8995/pd-neut-tof_ferrite-austenite-beer


View(html=<82,686 characters>, height=348)

The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [4]:
for parameter in project.free_parameters:
    parameter.free = False

In [5]:
project.display.pattern('expt_s2')

View(html=<482,857 characters>, height=480)

## Perform the analysis

The cells stay fixed: this is a calibration, and the two phases are the standards.

### Scales and time-of-flight calibration

Both phases' scales and each bank's DIFC offset put the peaks in place and at the right height.

In [6]:
bank_s2.linked_structures['ferrite'].scale.free = True
bank_s2.linked_structures['austenite'].scale.free = True
bank_s2.instrument.calib_d_to_tof_offset.free = True
bank_n2.instrument.calib_d_to_tof_offset.free = True

In [7]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,expt_n2,instrument,,calib_d_to_tof_offset,0,,-inf,inf,µs
2,expt_s2,instrument,,calib_d_to_tof_offset,0,,-inf,inf,µs
3,expt_s2,linked_structure,ferrite,scale,400,,-inf,inf,
4,expt_s2,linked_structure,austenite,scale,100,,-inf,inf,


In [8]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-tof_ferrite-austenite-beer, 2 banks (expt_n2, expt_s2), 5642 pts fitted / 5998 loaded, 4 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       41.43       229.76
      1      0.002     23.98        76.97   66.5% ↓
      2      0.003     23.84        76.05   1.2% ↓
      3      0.004     23.83        76.04   
      4      0.006     23.83        76.04   
  done · converged · 4 iters · reduced χ² 76.04 · Rwp 23.83 % · 0.01 s


View(html=<86,875 characters>, height=376)

In [9]:
project.display.pattern('expt_s2')

View(html=<483,067 characters>, height=480)

In [10]:
project.display.pattern('expt_n2')

View(html=<483,450 characters>, height=480)

### Background

Each bank's background points; the last, past the end of the data, stays fixed.

In [11]:
for point in bank_s2.background:
    point.intensity.free = True
for point in bank_n2.background:
    point.intensity.free = True
bank_s2.background[30].intensity.free = False
bank_n2.background[30].intensity.free = False

In [12]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-tof_ferrite-austenite-beer, 2 banks (expt_n2, expt_s2), 5642 pts fitted / 5998 loaded, 64 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       23.83        76.86
      1      0.039     18.05        44.10   42.6% ↓
      2      0.076     18.05        44.10   
  done · converged · 2 iters · reduced χ² 44.10 · Rwp 18.05 % · 0.09 s


View(html=<150,688 characters>, height=376)

In [13]:
project.display.pattern('expt_s2')

View(html=<545,832 characters>, height=480)

In [14]:
project.display.pattern('expt_n2')

View(html=<545,178 characters>, height=480)

### Peak widths

Each bank's widths.

In [15]:
bank_s2.peak.broad_gauss_sigma_0.free = True
bank_s2.peak.broad_gauss_sigma_1.free = True
bank_s2.peak.broad_gauss_sigma_2.free = True
bank_s2.peak.broad_lorentz_gamma_0.free = True
bank_n2.peak.broad_gauss_sigma_0.free = True
bank_n2.peak.broad_gauss_sigma_1.free = True
bank_n2.peak.broad_gauss_sigma_2.free = True
bank_n2.peak.broad_lorentz_gamma_0.free = True

In [16]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-tof_ferrite-austenite-beer, 2 banks (expt_n2, expt_s2), 5642 pts fitted / 5998 loaded, 72 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       18.05        44.16
      1      0.040     16.39        36.39   17.6% ↓
      2      0.046     16.38        36.35   0.1% ↓
      3      0.085     16.38        36.35   
  done · converged · 3 iters · reduced χ² 36.35 · Rwp 16.38 % · 0.10 s


View(html=<159,428 characters>, height=376)

In [17]:
project.display.pattern('expt_s2')

View(html=<545,584 characters>, height=480)

In [18]:
project.display.pattern('expt_n2')

View(html=<545,490 characters>, height=480)

### Displacement

Last, ferrite's displacement; austenite's follows it.

In [19]:
ferrite.atom_sites['Fe'].adp_iso.free = True

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: joint project pd-neut-tof_ferrite-austenite-beer, 2 banks (expt_n2, expt_s2), 5642 pts fitted / 5998 loaded, 73 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       16.38        36.36
      1      0.041      7.73         8.10   77.7% ↓
      2      0.048      7.14         6.91   14.7% ↓
      3      0.056      7.13         6.89   0.3% ↓
      4      0.064      7.13         6.89   
      5      0.102      7.13         6.89   
  done · converged · 5 iters · reduced χ² 6.89 · Rwp 7.13 % · 0.12 s


View(html=<160,739 characters>, height=376)

In [21]:
project.display.pattern('expt_s2')

View(html=<545,924 characters>, height=480)

In [22]:
project.display.pattern('expt_n2')

View(html=<545,717 characters>, height=480)

In [23]:
project.display.fit.correlations()

,1,2,3,4,5
1 expt_n2.peak.broad_gauss_sigma_2,,,,,
2 expt_n2.peak.broad_gauss_sigma_0,0.90,,,,
3 expt_n2.peak.broad_gauss_sigma_1,-0.98,-0.95,,,
4 expt_n2.background.1.intensity,0.00,0.01,-0.00,,
5 expt_n2.background.2.intensity,-0.01,-0.01,0.01,-0.83,


In [24]:
project.display.structure('ferrite')

View(html=<151,107 characters>, height=660)

In [25]:
project.display.structure('austenite')

View(html=<205,738 characters>, height=660)

## Save the project

In [26]:
project.save_as('pd-neut-tof_ferrite-austenite-beer')
project.report.save_html('pd-neut-tof_ferrite-austenite-beer.html')

Saving project to 'pd-neut-tof_ferrite-austenite-beer'
Saving report to 'pd-neut-tof_ferrite-austenite-beer.html'
